# Exploration du simulateur de sessions de recharge

Conception et test de la logique de génération : tirage de l'heure de
début (distribution pondérée), choix du connecteur, calcul de la durée
et de l'énergie, ajustement selon la température.

In [3]:
import random

POIDS_PAR_HEURE = {
    0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1,
    6: 2, 7: 3, 8: 3, 9: 2,
    10: 2, 11: 2, 12: 2, 13: 2, 14: 2, 15: 2,
    16: 4, 17: 10, 18: 12, 19: 12, 20: 8,
    21: 4, 22: 2, 23: 1,
}

def generer_heure_debut() -> tuple[int, int]:
    """Génère une heure et une minute de début de session, pondérées vers la soirée."""
    heures = list(POIDS_PAR_HEURE.keys())
    poids = list(POIDS_PAR_HEURE.values())
    heure = random.choices(heures, weights=poids, k=1)[0]
    minute = random.randint(0, 59)
    return heure, minute

for _ in range(5):
    print(generer_heure_debut())

(18, 7)
(23, 35)
(20, 5)
(10, 47)
(1, 14)


In [4]:
import duckdb
from pathlib import Path

ROOT_PATH = Path.cwd().resolve().parent
con = duckdb.connect(str(ROOT_PATH / "data" / "warehouse" / "electric_mobility.duckdb"))

connections_df = con.execute("SELECT * FROM connections").pl()
print(connections_df.shape)

(99, 10)


In [7]:
import polars as pl

connections_operationnelles = connections_df.filter(pl.col("is_operational") == True)
print(connections_operationnelles.shape)

(79, 10)


In [8]:
connecteur_tire = connections_operationnelles.sample(n=1)
connecteur_tire.to_dicts()[0]

{'connection_id': 331687,
 'poi_id': 198764,
 'power_kw': 7.0,
 'amps': None,
 'voltage': None,
 'connection_type': 'Type 2 (Socket Only)',
 'current_type': 'AC (Single-Phase)',
 'is_operational': True,
 'level_title': 'Level 2 : Medium (Over 2kW)',
 'is_fast_charge_capable': False}

In [9]:
import random
from datetime import date, timedelta

def generer_date_session(date_debut: date, date_fin: date) -> date:
    """Génère une date aléatoire entre date_debut et date_fin inclus."""
    nb_jours = (date_fin - date_debut).days
    decalage = random.randint(0, nb_jours)
    return date_debut + timedelta(days=decalage)

In [10]:
from datetime import date

for _ in range(5):
    print(generer_date_session(date(2026, 7, 20), date(2026, 7, 24)))

2026-07-22
2026-07-23
2026-07-21
2026-07-24
2026-07-20


In [27]:
date(2026, 7, 20)

datetime.date(2026, 7, 20)

In [11]:
from datetime import datetime, date

def combiner_date_heure(date_session: date, heure: int, minute: int) -> datetime:
    """Combine une date et une heure/minute en un objet datetime complet."""
    return datetime(date_session.year, date_session.month, date_session.day, heure, minute)

In [12]:
from datetime import date

dt = combiner_date_heure(date(2026, 7, 22), 7, 33)
print(dt)
print(dt.strftime("%Y-%m-%dT%H:00"))

2026-07-22 07:33:00
2026-07-22T07:00


In [15]:
meteo_df = con.execute("   SELECT temperature_2m FROM meteo WHERE poi_id = 198764 AND time = '2026-07-22T07:00'").pl()
print(meteo_df)

shape: (1, 1)
┌────────────────┐
│ temperature_2m │
│ ---            │
│ f64            │
╞════════════════╡
│ 17.4           │
└────────────────┘


In [16]:
def recuperer_temperature(con, poi_id: int, horodatage_tronque: str) -> float | None:
    """Récupère la température pour un poi_id et un horodatage donnés, ou None si absente."""
    resultat = con.execute(
        "SELECT temperature_2m FROM meteo WHERE poi_id = ? AND time = ?",
        [poi_id, horodatage_tronque]
    ).pl()
    if resultat.is_empty():
        return None
    return resultat.item()

In [17]:
temperature = recuperer_temperature(con, poi_id = 198764, horodatage_tronque = "2026-07-22T07:00") 
print(temperature)
print(type(temperature))

17.4
<class 'float'>


In [18]:
temperature = recuperer_temperature(con, poi_id = 999, horodatage_tronque = "2026-07-22T07:00") 
print(temperature)
print(type(temperature))

None
<class 'NoneType'>


In [19]:
import random

def generer_energie_cible(min_kwh: float = 5.0, max_kwh: float = 30.0) -> float:
    """Génère une énergie cible aléatoire, distribution uniforme."""
    return random.uniform(min_kwh, max_kwh)

In [20]:
for _ in range(5):
    print(generer_energie_cible())

19.48651551534011
5.605891727223865
9.299263124831986
17.87518784493365
27.434663383503388


In [22]:
def facteur_efficacite(temperature_c: float) -> float:
    """Renvoie un facteur d'efficacité de charge entre 0 et 1, réduit par le froid."""
    if temperature_c >= 20:
        return 1.0
    elif temperature_c >= 0:
        return 0.9
    else:
        return 0.75

In [23]:
import logging

logger = logging.getLogger(__name__)

def calculer_duree(energie_kwh: float, power_kw: float, temperature: float | None) -> float:
    """Calcule la durée de charge en heures, ajustée par la température."""
    if temperature is None:
        logger.warning("Température indisponible, utilisation d'une valeur par défaut (15°C).")
        temperature = 15.0

    efficacite = facteur_efficacite(temperature)
    return energie_kwh / (power_kw * efficacite)

In [24]:
duree = calculer_duree(energie_kwh=19.5, power_kw=7.0, temperature=17.4)
print(duree)

3.0952380952380953


In [25]:
duree = calculer_duree(energie_kwh=19.5, power_kw=7.0, temperature=-5.0)
print(duree)

3.7142857142857144


In [26]:
from datetime import datetime, timedelta

debut = datetime(2026, 7, 22, 7, 33)
duree = 3.0952380952380953

fin = debut + timedelta(hours=duree)
print(fin)

2026-07-22 10:38:42.857143


In [29]:
import polars as pl
import random
from datetime import date, timedelta, datetime
import logging

logger = logging.getLogger(__name__)

def generer_heure_debut() -> tuple[int, int]:
    """Génère une heure et une minute de début de session, pondérées vers la soirée."""
    heures = list(POIDS_PAR_HEURE.keys())
    poids = list(POIDS_PAR_HEURE.values())
    heure = random.choices(heures, weights=poids, k=1)[0]
    minute = random.randint(0, 59)
    return heure, minute

def generer_date_session(date_debut: date, date_fin: date) -> date:
    """Génère une date aléatoire entre date_debut et date_fin inclus."""
    nb_jours = (date_fin - date_debut).days
    decalage = random.randint(0, nb_jours)
    return date_debut + timedelta(days=decalage)

def combiner_date_heure(date_session: date, heure: int, minute: int) -> datetime:
    """Combine une date et une heure/minute en un objet datetime complet."""
    return datetime(date_session.year, date_session.month, date_session.day, heure, minute)

def recuperer_temperature(con, poi_id: int, horodatage_tronque: str) -> float | None:
    """Récupère la température pour un poi_id et un horodatage donnés, ou None si absente."""
    resultat = con.execute(
        "SELECT temperature_2m FROM meteo WHERE poi_id = ? AND time = ?",
        [poi_id, horodatage_tronque]
    ).pl()
    if resultat.is_empty():
        return None
    return resultat.item()

def generer_energie_cible(min_kwh: float = 5.0, max_kwh: float = 30.0) -> float:
    """Génère une énergie cible aléatoire, distribution uniforme."""
    return random.uniform(min_kwh, max_kwh)

def facteur_efficacite(temperature_c: float) -> float:
    """Renvoie un facteur d'efficacité de charge entre 0 et 1, réduit par le froid."""
    if temperature_c >= 20:
        return 1.0
    elif temperature_c >= 0:
        return 0.9
    else:
        return 0.75
        
def calculer_duree(energie_kwh: float, power_kw: float, temperature: float | None) -> float:
    """Calcule la durée de charge en heures, ajustée par la température."""
    if temperature is None:
        logger.warning("Température indisponible, utilisation d'une valeur par défaut (15°C).")
        temperature = 15.0

    efficacite = facteur_efficacite(temperature)
    return energie_kwh / (power_kw * efficacite)

def generer_session(
    con,
    connections_operationnelles: pl.DataFrame,
    date_debut: date,
    date_fin: date,
) -> dict:
    """Génère une session de recharge simulée complète."""

    # Tirer un connecteur
    connecteur = connections_operationnelles.sample(n=1).to_dicts()[0]

    # Générer heure/minute
    heure_debut, minute_debut = generer_heure_debut() 
    
    # Générer la date
    date_session = generer_date_session(date_debut, date_fin)

    # Combiner en debut
    date_heure_debut = combiner_date_heure(date_session, heure_debut, minute_debut) 

    # Récupérer la température
    poi_id = connecteur['poi_id']
    date_heure_debut_tronque = date_heure_debut.strftime("%Y-%m-%dT%H:00")
    
    temperature = recuperer_temperature(con, poi_id, date_heure_debut_tronque)

    # Générer l'énergie cible 
    energie_cible = generer_energie_cible(min_kwh = 5.0, max_kwh = 30.0) 
    
    # Calculer la durée
    puissance = connecteur['power_kw']
    duree = calculer_duree(energie_kwh = energie_cible, power_kw = puissance, temperature = temperature)

    # Calculer fin
    date_heure_fin = date_heure_debut + timedelta(hours=duree)

    # Renvoyer un dictionnaire avec toutes les colonnes de ta table sessions
    return {'connection_id' : connecteur['connection_id'],
            'debut' : date_heure_debut,
            'fin' : date_heure_fin, 
            'energie_kwh' : energie_cible
           }



connections_operationnelles = connections_df.filter(pl.col("is_operational") == True)
session_exemple = generer_session(
                                    con,
                                    connections_operationnelles,
                                    date_debut = date(2026, 7, 20),
                                    date_fin = date(2026, 7, 24),
                                ) 

print(session_exemple)

{'connection_id': 111739, 'debut': datetime.datetime(2026, 7, 20, 18, 19), 'fin': datetime.datetime(2026, 7, 20, 21, 27, 19, 444226), 'energie_kwh': 9.416203522018439}


In [30]:
sessions_exemple = [
    generer_session(con, connections_operationnelles, date(2026, 7, 20), date(2026, 7, 24))
    for _ in range(10)
]
for s in sessions_exemple:
    print(s)

{'connection_id': 331835, 'debut': datetime.datetime(2026, 7, 21, 18, 28), 'fin': datetime.datetime(2026, 7, 21, 20, 12, 30, 834421), 'energie_kwh': 12.193289152812813}
{'connection_id': 331873, 'debut': datetime.datetime(2026, 7, 21, 20, 10), 'fin': datetime.datetime(2026, 7, 21, 21, 11, 17, 9419), 'energie_kwh': 7.149740536103581}
{'connection_id': 331872, 'debut': datetime.datetime(2026, 7, 23, 5, 1), 'fin': datetime.datetime(2026, 7, 23, 6, 12, 36, 34604), 'energie_kwh': 7.518060557211301}
{'connection_id': 331938, 'debut': datetime.datetime(2026, 7, 23, 11, 42), 'fin': datetime.datetime(2026, 7, 23, 15, 43, 48, 577361), 'energie_kwh': 28.21112264639847}
{'connection_id': 111735, 'debut': datetime.datetime(2026, 7, 24, 17, 5), 'fin': datetime.datetime(2026, 7, 24, 17, 41, 58, 374281), 'energie_kwh': 13.556731718757602}
{'connection_id': 331835, 'debut': datetime.datetime(2026, 7, 22, 17, 3), 'fin': datetime.datetime(2026, 7, 22, 18, 34, 10, 11986), 'energie_kwh': 10.636134417159756

In [31]:
from simulation.sessions import generer_session

In [32]:
import duckdb
import polars as pl
from datetime import datetime

from warehouse.duckdb_loader import (
    creer_table_poi,
    inserer_poi,
    creer_table_connections,
    inserer_connections,
    creer_table_sessions,
    inserer_sessions,
)

con_test = duckdb.connect(":memory:")

# 1. Table poi, avec un POI de test
creer_table_poi(con_test)
poi_test = pl.DataFrame({
    "poi_id": [1],
    "title": ["Test"],
    "town": ["Paris"],
    "town_normalisee": ["Paris"],
    "postcode": ["75001"],
    "latitude": [48.85],
    "longitude": [2.35],
    "number_of_points": [1],
    "usage_cost": ["Free"],
    "date_last_confirmed": ["2026-01-01"],
})
inserer_poi(con_test, poi_test)

# 2. Table connections, avec un connecteur de test
creer_table_connections(con_test)
connection_test = pl.DataFrame({
    "connection_id": [1],
    "poi_id": [1],
    "power_kw": [7.0],
    "amps": [32],
    "voltage": [400.0],
    "connection_type": ["Type 2 (Socket Only)"],
    "current_type": ["AC (Single-Phase)"],
    "is_operational": [True],
    "level_title": ["Level 2 : Medium (Over 2kW)"],
    "is_fast_charge_capable": [False],
})
inserer_connections(con_test, connection_test)

# 3. Table sessions
creer_table_sessions(con_test)

sessions_test = pl.DataFrame({
    "connection_id": [1],
    "debut": [datetime(2026, 7, 22, 18, 30)],
    "fin": [datetime(2026, 7, 22, 20, 0)],
    "energie_kwh": [15.0],
})
inserer_sessions(con_test, sessions_test)

resultat = con_test.execute("SELECT * FROM sessions").fetchall()
print(resultat)

[(1, 1, datetime.datetime(2026, 7, 22, 18, 30), datetime.datetime(2026, 7, 22, 20, 0), 15.0)]
